# Waymaker Artemis — Mission 005-Y: 고르는 질문의 부분마다 자기 코드 확인

무료 Colab T4에서 저장소 코드(커밋 `37f1938a9d807f96aaf1fefe78c5d5089b81ec5f`)로 공식 `google/gemma-4-E4B-it`(revision `ee0ef60…`)를 NF4로 **한 번** 올립니다.

1. 새로 잠근 합성 수용 집합 81건(계약 v18)을 005-Y 파이프라인으로 한 번 처리합니다. 고르는 질문은 005-X처럼 코드마다 따로 묻습니다. 한 부분의 답이 자기 코드를 쓰지 않으면 한 번 더 묻고, 그래도 없으면 근거 문장을 인용한 안내를 붙입니다. 두 부분의 답이 같으면 한 번만 잇습니다.
2. `synthetic_safety_gate_v20`이 원 모델·추론 전·끝단 결과를 따로 채점해 `pass`·`fail`·`blocked`를 정합니다. 원 모델은 처음 답으로, 끝단은 사용자에게 나간 답으로 채점합니다.
3. 게이트가 `pass`일 때만 DEV-20 v2를 같은 파이프라인과 v2 기록부로 한 번 처리합니다.

- 계약·평가기·수용 집합·문턱값은 이 실행 전에 고정했습니다(`validation/mission005y_acceptance_manifest.json`).
- held-out 40건은 실행하지 않습니다. Official Gold Accuracy는 `N/A`입니다. 비용은 무료 런타임뿐입니다.


In [ ]:
# 1. GPU·VRAM·디스크 확인 (읽기 전용)
import shutil, subprocess, sys
print(sys.version)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
print('Disk free GB:', round(shutil.disk_usage('/content').free / 1e9, 2))

## 2. 무료 실행 환경 확인

1. **런타임 → 런타임 유형 변경**에서 `T4 GPU`인지 봅니다.
2. **리소스 보기**에서 구독 없음·컴퓨팅 단위 미사용 상태인지 봅니다.
3. 둘 다 맞으면 아래 값을 `True`로 바꿉니다.

In [ ]:
FREE_TIER_CONFIRMED = False  # 위 두 가지를 직접 확인한 뒤에만 True로 바꿉니다
assert FREE_TIER_CONFIRMED, '무료 T4 런타임인지 먼저 확인하세요'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
assert gpu == 'Tesla T4', f'예상한 무료 T4가 아닙니다: {gpu!r}'

In [ ]:
# 3. 코드와 패키지 버전 고정
CODE_COMMIT = '37f1938a9d807f96aaf1fefe78c5d5089b81ec5f'
!git init -q /content/Visable
!git -C /content/Visable remote add origin https://github.com/club-paradiso/Visable.git
# artemis_eval -> procedure_route imports backend/services (standard library only, about 5MB).
!git -C /content/Visable sparse-checkout set experiments/waymaker-ai backend/services
!git -C /content/Visable fetch -q --depth 1 --filter=blob:none origin {CODE_COMMIT}
!git -C /content/Visable checkout -q FETCH_HEAD
head = subprocess.run(['git', '-C', '/content/Visable', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == CODE_COMMIT, f'코드 커밋 불일치: {head}'
# Colab 기본 torch(2026-11-09 기준 2.14.1)는 감사한 실행과 다릅니다. torch를 불러오기 전에 고정한 torch를 설치합니다.
%pip install -q "torch==2.11.0+cu130" "torchvision==0.26.0+cu130" "torchaudio==2.11.0+cu130" --index-url https://download.pytorch.org/whl/cu130
%pip install -q "transformers==5.20.2" "bitsandbytes==0.50.2"
import importlib.metadata as md
print({n: md.version(n) for n in ('torch', 'transformers', 'bitsandbytes', 'accelerate', 'huggingface-hub', 'tokenizers')})

In [ ]:
# 4. 체크포인트 다운로드(이 셀만 가중치를 받습니다, 약 16GB)와 SHA-256 확인
import json, os
REPO = '/content/Visable/experiments/waymaker-ai'
sys.path[:0] = [REPO, REPO + '/artemis_bootstrap']
policy = json.load(open(REPO + '/configs/artemis_cuda_runtime.json'))
MODEL, REVISION = policy['model']['checkpoint'], policy['model']['revision']
assert (MODEL, REVISION) == ('google/gemma-4-E4B-it', 'ee0ef6023621cff504d758262d4e04895a5af4a2')
assert shutil.disk_usage('/content').free / 1e9 >= policy['limits']['min_free_disk_gb_for_download']
from huggingface_hub import snapshot_download
from pathlib import Path
snapshot = Path(snapshot_download(MODEL, revision=REVISION, token=False))
from waymaker_ai import cuda_policy
verified = cuda_policy.validate_snapshot(snapshot)  # 1~2분
print(verified['weight_integrity'], verified['artifact_integrity']['model.safetensors']['digest'])

In [ ]:
# 5. CUDA 사전 검사. 이후로는 네트워크를 쓰지 않습니다.
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import gemma4_cuda_runtime as cuda
check = cuda.preflight()
print('blockers:', check['blockers'], '| version drift:', check['probe']['version_drift'])

In [ ]:
# 6. 모델을 한 번만 올리고 NF4 검증
prepared = cuda.PreparedRun()
print('loaded:', prepared.prepare(), '| blocker:', prepared.blocker)
quant = prepared.header.get('load', {}).get('quantization', {})
print({k: quant.get(k) for k in ('passed', 'quantized_decoder_layers', 'linear4bit_modules', 'quant_types', 'parameter_devices')})

In [ ]:
# 7. 잠긴 합성 수용 집합 (계약 v18)과 게이트 v20 판단
from pathlib import Path
import synthetic_safety_gate_v20 as gate20
ACC_DIR = Path('/content/artemis_mission005y_acceptance')
record = gate20.run(prepared, ACC_DIR)
print(json.dumps({k: record.get(k) for k in ('decision', 'fail_reasons', 'blocked_reasons', 'summary_counts')}, ensure_ascii=False, indent=1))


In [ ]:
# 8. DEV-20 v2: 게이트가 pass일 때만. dev20_v2.run이 봉인된 통과 기록을 다시 채점해 확인하고, 아니면 거부합니다.
import dev20_v2
DEV_DIR = Path('/content/artemis_mission005y_dev20_v2')
if record['decision'] == 'pass':
    print(dev20_v2.run(prepared, DEV_DIR, ACC_DIR / 'gate_decision.json'))
    result = dev20_v2.score(DEV_DIR)
    (DEV_DIR / 'dev20_v2_results.json').write_text(json.dumps(result, ensure_ascii=False, indent=1) + '\n')
    print(json.dumps(result['summary'], ensure_ascii=False, indent=1))
else:
    print('게이트가', record['decision'], '이므로 DEV-20 v2를 실행하지 않습니다.')


In [ ]:
# 9. 결과 해시
import hashlib, os
for d in (ACC_DIR, DEV_DIR):
    for name in ('generations.jsonl', 'sessions.jsonl', 'pipeline.jsonl', 'gate_decision.json', 'dev20_v2_results.json'):
        p = d / name
        if p.exists():
            raw = p.read_bytes()
            print(d.name, name, len(raw), hashlib.sha256(raw).hexdigest())


In [ ]:
# 10. 결과를 바로 브라우저 페이지로 넘깁니다(런타임이 회수되면 /content가 사라집니다)
import base64, gzip
from IPython.display import Javascript, display
F = {f'{d.name}/{p.name}': p.read_bytes() for d in (ACC_DIR, DEV_DIR) if d.exists() for p in sorted(d.glob('*'))}
P = {k: base64.urlsafe_b64encode(gzip.compress(v, mtime=0)).decode() for k, v in F.items()}
display(Javascript('const P=' + json.dumps(P) + ';for(const [k,v] of Object.entries(P)){window.top.postMessage({xfer:1,k:k,v:v},"*")}'))
print({k: [len(v), hashlib.sha256(v).hexdigest()] for k, v in F.items()})


## 끝난 뒤

- 각 폴더의 `generations.jsonl`, `sessions.jsonl`, `pipeline.jsonl`을 고치지 않고 `experiments/waymaker-ai/artemis_bootstrap/runs/mission005y/` 아래에 넣습니다.
- 저장소에서 `python3 artemis_bootstrap/synthetic_safety_gate_v20.py score --out-dir <수용 폴더>`로 판정을 다시 계산해 Colab 결과와 대조합니다.
- **런타임 → 런타임 연결 해제 및 삭제**로 GPU를 반납합니다.
